In [0]:
dbutils.widgets.text("fecha_corte", "2026-11-05")
dbutils.widgets.dropdown("modo", "incremental", ["incremental", "full", "reproceso"])
dbutils.widgets.dropdown("env", "dev", ["dev", "qa", "prod"])
dbutils.widgets.text("run_id", "")

In [0]:
import sys, os
sys.path.append(os.path.abspath(".."))

from src.common.config import PipelineConfig
from src.common.control import ejecucion
from src.silver.proceso import procesar_todo

params = dict(
    fecha_corte=dbutils.widgets.get("fecha_corte"),
    modo=dbutils.widgets.get("modo"),
    env=dbutils.widgets.get("env"),
)
if dbutils.widgets.get("run_id"):
    params["run_id"] = dbutils.widgets.get("run_id")

cfg = PipelineConfig(**params)

with ejecucion(spark, cfg, "silver"):
    resultado = procesar_todo(spark, cfg)

display(spark.createDataFrame(resultado))

In [0]:
%sql
SELECT tabla,
       filas_leidas     AS bronze,
       filas_escritas   AS silver,
       filas_rechazadas AS cuarentena,
       filas_leidas - filas_escritas - filas_rechazadas AS diferencia,
       totales_monetarios
FROM workspace.default.ctl_cargas
WHERE capa = 'silver'
  AND run_id = (SELECT max(run_id) FROM workspace.default.ctl_cargas WHERE capa = 'silver')
ORDER BY tabla;

In [0]:
%sql
SELECT dataset, regla, dimension, severidad, accion, fallidos, pct_fallidos, umbral_pct, resultado
FROM workspace.default.ctl_calidad_resultados
WHERE run_id = (SELECT max(run_id) FROM workspace.default.ctl_calidad_resultados)
  AND fallidos > 0
ORDER BY dataset, fallidos DESC;

In [0]:
%sql
SELECT poliza_id, prima_raw, fecha_inicio_vigencia_raw, fecha_fin_vigencia_raw,
       _motivos, _pipeline, _cuarentena_run_id, _source_file
FROM workspace.default.quarantine_polizas;